# DATA_STORM — Grupo 7 — Smart Grids

**Disciplina:** Análise de Dados — ANALDA-101002  
**Projeto:** Engenharia de Decisão e Inteligência em Escala  
**Tema do Grupo 7:** Soberania de Dados e IA em Redes de Energia Inteligentes (Smart Grids)

## Integrantes

- Estela Argolo
- William Henrique
- Rodrigo Madureira

## Objetivo executivo

Construir uma solução completa de Business Analytics no Google Colab para prever a estabilidade de uma rede elétrica inteligente, integrando:

1. Pipeline ETL;
2. Modelagem dimensional em Star Schema;
3. Análise Exploratória de Dados (EDA);
4. Modelagem Preditiva;
5. Business Insights e simulação OLAP/BI.

## 1. Problema de negócio

Smart Grids combinam sensores, comunicação e controle descentralizado para equilibrar produção e consumo de energia. O desafio é prever instabilidade antes que ela cause falhas operacionais.

**Pergunta central:**  
> É possível prever automaticamente se uma Smart Grid está estável ou instável a partir de variáveis operacionais de tempo de resposta, potência e elasticidade?

**Impacto esperado:**  
- reduzir risco de indisponibilidade;
- antecipar cenários críticos;
- apoiar manutenção preventiva;
- melhorar confiabilidade de infraestrutura crítica;
- permitir decisões orientadas por dados.

## 2. Dataset externo

Fonte: **UCI Machine Learning Repository — Electrical Grid Stability Simulated Data**.

O dataset contém 10.000 simulações de uma rede elétrica de 4 nós, com:

- tempos de reação (`tau1` a `tau4`);
- potências nominais (`p1` a `p4`);
- coeficientes de elasticidade/preço (`g1` a `g4`);
- valor contínuo de estabilidade (`stab`);
- classe final (`stabf`: stable/unstable).

A ingestão será feita diretamente por URL pública, com persistência dos dados brutos e tratados.

In [ ]:
# Instalação de dependências adicionais, se necessário
import sys
import subprocess

try:
    import seaborn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn", "-q"])

In [ ]:
# Imports principais
from pathlib import Path
from datetime import datetime
import io
import zipfile
import time
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, mean_squared_error, r2_score
)
from sklearn.inspection import permutation_importance
import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 3. Configuração do ambiente e persistência

No Google Colab, o notebook tentará montar o Google Drive e criar a estrutura de pastas do projeto. Se estiver rodando localmente, criará a estrutura na pasta atual.

In [ ]:
PROJECT_NAME = "DATA_STORM_GRUPO_7_SMART_GRIDS"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive') / PROJECT_NAME
    AMBIENTE = "Google Colab + Google Drive"
except Exception:
    BASE_DIR = Path.cwd() / PROJECT_NAME
    AMBIENTE = "Ambiente local"

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
FIGURES_DIR = BASE_DIR / "outputs" / "figures"
MODELS_DIR = BASE_DIR / "outputs" / "models"

for path in [RAW_DIR, PROCESSED_DIR, FIGURES_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Ambiente detectado: {AMBIENTE}")
print(f"Diretório do projeto: {BASE_DIR}")

## 4. Camada Bronze — Extract

A camada Bronze contém os dados brutos, exatamente como foram obtidos da fonte externa.

In [ ]:
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/471/electrical+grid+stability+simulated+data.zip"
FALLBACK_CSV_URL = "https://raw.githubusercontent.com/rahatUlAin/Electrical-Grid-Stability-Simulated-Data-/master/Data_for_UCI_named.csv"


def carregar_dataset_smart_grid() -> pd.DataFrame:
    """Carrega o dataset oficial da UCI, com fallback em CSV público."""
    try:
        response = requests.get(UCI_ZIP_URL, timeout=30)
        response.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            csv_files = [name for name in zf.namelist() if name.lower().endswith('.csv')]
            if not csv_files:
                raise FileNotFoundError("Nenhum arquivo CSV encontrado no ZIP da UCI.")
            with zf.open(csv_files[0]) as file:
                df = pd.read_csv(file)
        fonte = UCI_ZIP_URL
    except Exception as erro:
        print(f"Falha ao baixar pela UCI, usando fallback público. Detalhe: {erro}")
        df = pd.read_csv(FALLBACK_CSV_URL)
        fonte = FALLBACK_CSV_URL

    return df, fonte

bronze_df, fonte_utilizada = carregar_dataset_smart_grid()
bronze_df.to_csv(RAW_DIR / "smart_grid_raw.csv", index=False)

print("Fonte utilizada:", fonte_utilizada)
print("Dimensão da camada Bronze:", bronze_df.shape)
display(bronze_df.head())

## 5. Camada Silver — Transform

Tratamentos aplicados:

- padronização de nomes de colunas;
- remoção de duplicatas;
- tratamento de nulos;
- conversão de tipos;
- criação de variáveis derivadas;
- criação do alvo binário `is_unstable`.

In [ ]:
def transformar_smart_grid(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.drop_duplicates()

    # Garantia de colunas esperadas
    expected_cols = [
        "tau1", "tau2", "tau3", "tau4",
        "p1", "p2", "p3", "p4",
        "g1", "g2", "g3", "g4",
        "stab", "stabf"
    ]
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas ausentes no dataset: {missing}")

    numeric_cols = [c for c in expected_cols if c != "stabf"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["stabf"] = df["stabf"].astype(str).str.lower().str.strip()
    df = df.dropna()
    df = df[df["stabf"].isin(["stable", "unstable"])]

    tau_cols = ["tau1", "tau2", "tau3", "tau4"]
    g_cols = ["g1", "g2", "g3", "g4"]

    df["tempo_resposta_medio"] = df[tau_cols].mean(axis=1)
    df["tempo_resposta_max"] = df[tau_cols].max(axis=1)
    df["potencia_total_consumo"] = df[["p2", "p3", "p4"]].abs().sum(axis=1)
    df["potencia_produtor"] = df["p1"]
    df["elasticidade_media"] = df[g_cols].mean(axis=1)
    df["elasticidade_max"] = df[g_cols].max(axis=1)
    df["is_unstable"] = (df["stabf"] == "unstable").astype(int)

    # Índice sintético para ordenação de risco operacional.
    # Não substitui o target oficial; serve para BI e priorização.
    df["grid_risk_score"] = (
        df["tempo_resposta_medio"].rank(pct=True) * 0.35
        + df["tempo_resposta_max"].rank(pct=True) * 0.25
        + df["potencia_total_consumo"].rank(pct=True) * 0.20
        + df["elasticidade_media"].rank(pct=True) * 0.20
    )

    df.insert(0, "simulation_id", np.arange(1, len(df) + 1))
    return df

silver_df = transformar_smart_grid(bronze_df)
silver_df.to_csv(PROCESSED_DIR / "smart_grid_silver.csv", index=False)

print("Dimensão da camada Silver:", silver_df.shape)
display(silver_df.head())

## 6. Camada Gold — Load e Star Schema

A camada Gold organiza os dados em tabelas dimensionais e tabela fato, permitindo análise OLAP e consultas analíticas.

In [ ]:
def construir_dimensao(df: pd.DataFrame, colunas: list, chave: str) -> pd.DataFrame:
    dim = df[colunas].drop_duplicates().reset_index(drop=True)
    dim.insert(0, chave, np.arange(1, len(dim) + 1))
    return dim

extraction_timestamp = datetime.utcnow().isoformat(timespec="seconds") + "Z"

dim_simulation = pd.DataFrame({
    "simulation_id": silver_df["simulation_id"],
    "source_dataset": "UCI Electrical Grid Stability Simulated Data",
    "source_url": fonte_utilizada,
    "extraction_timestamp": extraction_timestamp,
})

dim_power_profile = construir_dimensao(
    silver_df,
    ["p1", "p2", "p3", "p4", "potencia_total_consumo", "potencia_produtor"],
    "power_profile_id"
)

dim_response_profile = construir_dimensao(
    silver_df,
    ["tau1", "tau2", "tau3", "tau4", "tempo_resposta_medio", "tempo_resposta_max"],
    "response_profile_id"
)

dim_elasticity_profile = construir_dimensao(
    silver_df,
    ["g1", "g2", "g3", "g4", "elasticidade_media", "elasticidade_max"],
    "elasticity_profile_id"
)

dim_stability_class = pd.DataFrame({
    "stability_class_id": [1, 2],
    "stability_label": ["stable", "unstable"],
    "is_unstable": [0, 1],
    "risk_level": ["Operação normal", "Risco operacional"]
})

fact_grid_stability = silver_df.merge(
    dim_power_profile,
    on=["p1", "p2", "p3", "p4", "potencia_total_consumo", "potencia_produtor"],
    how="left"
).merge(
    dim_response_profile,
    on=["tau1", "tau2", "tau3", "tau4", "tempo_resposta_medio", "tempo_resposta_max"],
    how="left"
).merge(
    dim_elasticity_profile,
    on=["g1", "g2", "g3", "g4", "elasticidade_media", "elasticidade_max"],
    how="left"
).merge(
    dim_stability_class[["stability_class_id", "stability_label"]],
    left_on="stabf",
    right_on="stability_label",
    how="left"
)

fact_grid_stability = fact_grid_stability[[
    "simulation_id", "power_profile_id", "response_profile_id", "elasticity_profile_id",
    "stability_class_id", "stab", "grid_risk_score"
]].copy()
fact_grid_stability.insert(0, "fact_id", np.arange(1, len(fact_grid_stability) + 1))
fact_grid_stability = fact_grid_stability.rename(columns={"stab": "stab_value"})

gold_tables = {
    "dim_simulation": dim_simulation,
    "dim_power_profile": dim_power_profile,
    "dim_response_profile": dim_response_profile,
    "dim_elasticity_profile": dim_elasticity_profile,
    "dim_stability_class": dim_stability_class,
    "fact_grid_stability": fact_grid_stability,
}

for table_name, table in gold_tables.items():
    table.to_csv(PROCESSED_DIR / f"{table_name}.csv", index=False)
    print(f"{table_name}: {table.shape}")

print("Tabelas Gold salvas em:", PROCESSED_DIR)
display(fact_grid_stability.head())

## 7. Desenho lógico do Data Warehouse

```mermaid
erDiagram
    DIM_SIMULATION ||--o{ FACT_GRID_STABILITY : simulation_id
    DIM_POWER_PROFILE ||--o{ FACT_GRID_STABILITY : power_profile_id
    DIM_RESPONSE_PROFILE ||--o{ FACT_GRID_STABILITY : response_profile_id
    DIM_ELASTICITY_PROFILE ||--o{ FACT_GRID_STABILITY : elasticity_profile_id
    DIM_STABILITY_CLASS ||--o{ FACT_GRID_STABILITY : stability_class_id
```

## 8. EDA — Estatística descritiva

Nesta etapa calculamos medidas de tendência central, dispersão, forma da distribuição e análise de outliers.

In [ ]:
numeric_cols = silver_df.select_dtypes(include=[np.number]).columns.tolist()
eda_cols = [c for c in numeric_cols if c not in ["simulation_id", "is_unstable"]]

stats = []
for col in eda_cols:
    serie = silver_df[col]
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    stats.append({
        "variavel": col,
        "media": serie.mean(),
        "mediana": serie.median(),
        "moda": serie.mode().iloc[0] if not serie.mode().empty else np.nan,
        "variancia": serie.var(),
        "desvio_padrao": serie.std(),
        "amplitude": serie.max() - serie.min(),
        "iqr": iqr,
        "assimetria_skewness": serie.skew(),
        "curtose": serie.kurtosis(),
        "outliers_iqr": ((serie < (q1 - 1.5 * iqr)) | (serie > (q3 + 1.5 * iqr))).sum()
    })

stats_df = pd.DataFrame(stats).sort_values("variavel")
stats_df.to_csv(PROCESSED_DIR / "eda_estatistica_descritiva.csv", index=False)
display(stats_df)

## 9. Visualizações — Distribuição, boxplot e target

In [ ]:
plot_cols = ["tempo_resposta_medio", "tempo_resposta_max", "potencia_total_consumo", "elasticidade_media", "grid_risk_score", "stab"]

for col in plot_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(silver_df[col], kde=True)
    plt.title(f"Distribuição — {col}")
    plt.xlabel(col)
    plt.ylabel("Frequência")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"hist_{col}.png", dpi=150)
    plt.show()

for col in plot_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x="stabf", y=col, data=silver_df)
    plt.title(f"Boxplot de {col} por classe de estabilidade")
    plt.xlabel("Classe")
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"boxplot_{col}_por_stabf.png", dpi=150)
    plt.show()

## 10. Correlação e análise multivariada

In [ ]:
correlation_cols = [
    "tau1", "tau2", "tau3", "tau4",
    "p1", "p2", "p3", "p4",
    "g1", "g2", "g3", "g4",
    "tempo_resposta_medio", "tempo_resposta_max",
    "potencia_total_consumo", "elasticidade_media",
    "grid_risk_score", "stab", "is_unstable"
]

corr = silver_df[correlation_cols].corr(method="pearson")
corr.to_csv(PROCESSED_DIR / "eda_matriz_correlacao.csv")

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de Correlação — Smart Grid")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "heatmap_correlacao.png", dpi=150)
plt.show()

target_corr = corr["is_unstable"].sort_values(key=abs, ascending=False).to_frame("correlacao_com_target")
display(target_corr)

## 11. Conclusões para a modelagem

Principais decisões técnicas:

- Variáveis de tempo de resposta e elasticidade tendem a apresentar relação importante com instabilidade.
- Como há relações potencialmente não lineares, modelos baseados em árvore são candidatos fortes.
- O target de classificação será `is_unstable`.
- A variável `stab` não será usada como feature na classificação para evitar vazamento de informação, pois ela é diretamente relacionada à classe `stabf`.
- As variáveis numéricas podem beneficiar modelos lineares e redes neurais quando padronizadas.

## 12. Modelagem Preditiva — Classificação

Comparativo de modelos:

- Logistic Regression;
- Decision Tree;
- Random Forest;
- Neural Network (MLP).

Métricas utilizadas: Accuracy, Precision, Recall, F1-Score e ROC-AUC.

In [ ]:
FEATURES = [
    "tau1", "tau2", "tau3", "tau4",
    "p1", "p2", "p3", "p4",
    "g1", "g2", "g3", "g4",
    "tempo_resposta_medio", "tempo_resposta_max",
    "potencia_total_consumo", "potencia_produtor",
    "elasticidade_media", "elasticidade_max",
    "grid_risk_score",
]

X = silver_df[FEATURES]
y = silver_df["is_unstable"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=250, max_depth=None, random_state=42, class_weight="balanced", n_jobs=-1),
    "Neural Network (MLP)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42))
    ])
}

classification_results = []
fitted_models = {}

for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    classification_results.append({
        "modelo": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "tempo_execucao_s": elapsed
    })
    fitted_models[name] = model

classification_results_df = pd.DataFrame(classification_results).sort_values("f1", ascending=False)
classification_results_df.to_csv(PROCESSED_DIR / "modelos_classificacao_metricas.csv", index=False)
display(classification_results_df)

champion_name = classification_results_df.iloc[0]["modelo"]
champion_model = fitted_models[champion_name]
print("Modelo campeão:", champion_name)

In [ ]:
y_pred_champion = champion_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_champion)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Matriz de Confusão — {champion_name}")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "matriz_confusao_modelo_campeao.png", dpi=150)
plt.show()

print(classification_report(y_test, y_pred_champion, target_names=["stable", "unstable"]))

## 13. Importância de variáveis e teste de estresse

O teste de estresse remove a variável mais importante do Random Forest para verificar a estabilidade do modelo.

In [ ]:
rf_model = fitted_models["Random Forest"]
rf_importances = pd.DataFrame({
    "feature": FEATURES,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

rf_importances.to_csv(PROCESSED_DIR / "feature_importance_random_forest.csv", index=False)
display(rf_importances.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(data=rf_importances.head(10), x="importance", y="feature")
plt.title("Top 10 variáveis mais importantes — Random Forest")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_importance_random_forest.png", dpi=150)
plt.show()

feature_removida = rf_importances.iloc[0]["feature"]
features_stress = [f for f in FEATURES if f != feature_removida]

X_train_s, X_test_s = X_train[features_stress], X_test[features_stress]
stress_model = RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced", n_jobs=-1)
start = time.perf_counter()
stress_model.fit(X_train_s, y_train)
stress_time = time.perf_counter() - start
stress_pred = stress_model.predict(X_test_s)

stress_result = pd.DataFrame([
    {
        "cenario": "Modelo original Random Forest",
        "feature_removida": "nenhuma",
        "f1": classification_results_df.loc[classification_results_df["modelo"] == "Random Forest", "f1"].iloc[0],
        "tempo_execucao_s": classification_results_df.loc[classification_results_df["modelo"] == "Random Forest", "tempo_execucao_s"].iloc[0]
    },
    {
        "cenario": "Teste de estresse sem variável mais importante",
        "feature_removida": feature_removida,
        "f1": f1_score(y_test, stress_pred, zero_division=0),
        "tempo_execucao_s": stress_time
    }
])
stress_result["variacao_f1_percentual"] = stress_result["f1"].pct_change() * 100
stress_result.to_csv(PROCESSED_DIR / "teste_estresse_random_forest.csv", index=False)
display(stress_result)

## 14. Modelagem Preditiva — Regressão

Além da classificação, também treinamos modelos para prever `stab`, a medida contínua de estabilidade. Métricas: RMSE e R².

In [ ]:
y_reg = silver_df["stab"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.25, random_state=42)

reg_models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=250, random_state=42, n_jobs=-1)
}

reg_results = []
for name, model in reg_models.items():
    start = time.perf_counter()
    model.fit(X_train_r, y_train_r)
    elapsed = time.perf_counter() - start
    pred = model.predict(X_test_r)
    rmse = mean_squared_error(y_test_r, pred, squared=False)
    reg_results.append({
        "modelo": name,
        "rmse": rmse,
        "r2": r2_score(y_test_r, pred),
        "tempo_execucao_s": elapsed
    })

reg_results_df = pd.DataFrame(reg_results).sort_values("rmse")
reg_results_df.to_csv(PROCESSED_DIR / "modelos_regressao_metricas.csv", index=False)
display(reg_results_df)

## 15. Simulação OLAP/BI e Dashboard de Decisão

A camada BI transforma os dados técnicos em visão executiva.

In [ ]:
bi_df = silver_df.copy()
bi_df["faixa_tempo_resposta"] = pd.qcut(bi_df["tempo_resposta_medio"], q=4, labels=["Muito baixo", "Baixo", "Alto", "Muito alto"])
bi_df["faixa_elasticidade"] = pd.qcut(bi_df["elasticidade_media"], q=4, labels=["Muito baixa", "Baixa", "Alta", "Muito alta"])
bi_df["faixa_risco"] = pd.qcut(bi_df["grid_risk_score"], q=4, labels=["Baixo", "Moderado", "Alto", "Crítico"])

kpis = pd.DataFrame({
    "indicador": [
        "Total de simulações",
        "Cenários instáveis",
        "% de instabilidade",
        "Stab médio",
        "Score médio de risco"
    ],
    "valor": [
        len(bi_df),
        int(bi_df["is_unstable"].sum()),
        f"{bi_df['is_unstable'].mean() * 100:.2f}%",
        round(bi_df["stab"].mean(), 4),
        round(bi_df["grid_risk_score"].mean(), 4)
    ]
})

olap_risco = pd.pivot_table(
    bi_df,
    values="is_unstable",
    index="faixa_tempo_resposta",
    columns="faixa_elasticidade",
    aggfunc="mean"
) * 100

risco_por_faixa = bi_df.groupby("faixa_risco", observed=True).agg(
    total_simulacoes=("simulation_id", "count"),
    taxa_instabilidade_percentual=("is_unstable", lambda x: x.mean() * 100),
    stab_medio=("stab", "mean"),
    risco_medio=("grid_risk_score", "mean")
).reset_index()

kpis.to_csv(PROCESSED_DIR / "bi_kpis.csv", index=False)
olap_risco.to_csv(PROCESSED_DIR / "bi_olap_tempo_vs_elasticidade.csv")
risco_por_faixa.to_csv(PROCESSED_DIR / "bi_risco_por_faixa.csv", index=False)

display(kpis)
display(olap_risco)
display(risco_por_faixa)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=risco_por_faixa, x="faixa_risco", y="taxa_instabilidade_percentual")
plt.title("Taxa de Instabilidade por Faixa de Risco")
plt.xlabel("Faixa de risco")
plt.ylabel("Instabilidade (%)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dashboard_taxa_instabilidade_por_risco.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
sns.heatmap(olap_risco, annot=True, fmt=".1f", cmap="Reds")
plt.title("OLAP — % Instabilidade por Tempo de Resposta x Elasticidade")
plt.xlabel("Faixa de elasticidade")
plt.ylabel("Faixa de tempo de resposta")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dashboard_olap_instabilidade.png", dpi=150)
plt.show()

## 16. Recomendações estratégicas

1. **Monitorar tempo de resposta médio e máximo:** cenários com maior tempo de reação tendem a elevar o risco operacional.
2. **Usar o modelo campeão como alerta preventivo:** o modelo pode funcionar como camada de decisão antes de falhas reais.
3. **Priorizar cenários críticos no dashboard:** faixas com maior taxa de instabilidade devem receber atenção operacional.
4. **Evitar dependência de uma única variável:** o teste de estresse mostra se o modelo continua robusto quando a principal feature é removida.
5. **Evolução futura:** conectar sensores IoT reais, streaming de dados e alerta automático para operação.

## 17. Conclusão executiva

O Grupo 7 demonstrou uma solução completa de Business Analytics para Smart Grids. O pipeline começa na ingestão de dados externos, passa por tratamento e modelagem dimensional, realiza análise estatística, treina modelos preditivos e finaliza com indicadores de decisão.

A solução tem potencial de apoiar operadores de energia em decisões preventivas, reduzindo risco de instabilidade e aumentando a confiabilidade da rede elétrica inteligente.

In [ ]:
# Persistência do modelo campeão e resumo final
joblib.dump(champion_model, MODELS_DIR / "modelo_campeao_classificacao.pkl")

resumo_final = {
    "projeto": PROJECT_NAME,
    "grupo": "Grupo 7",
    "tema": "Smart Grids",
    "integrantes": ["Estela Argolo", "William Henrique", "Rodrigo Madureira"],
    "modelo_campeao": champion_name,
    "melhor_f1": float(classification_results_df.iloc[0]["f1"]),
    "melhor_accuracy": float(classification_results_df.iloc[0]["accuracy"]),
    "fonte_dados": fonte_utilizada,
    "diretorio_saida": str(BASE_DIR)
}

pd.Series(resumo_final).to_json(PROCESSED_DIR / "resumo_final.json", force_ascii=False, indent=2)
resumo_final